In [1]:
# ============================================================
# CREATE METADATA-ONLY SUBSET FOR VINBIGDATA CHEST X-RAY
# 500 Normal + All Abnormal Images
# No image copy, no DICOM zip
# ============================================================

import os
import random
import pandas as pd
from pathlib import Path

# ============================================================
# 1. CONFIG
# ============================================================

SEED = 42
N_NORMAL = 500

NO_FINDING_CLASS_ID = 14
NO_FINDING_CLASS_NAME = "No finding"

# Kaggle dataset path
DATASET_DIR = Path("/kaggle/input/vinbigdata-chest-xray-abnormalities-detection")

# Fallback path if Kaggle mounts competition dataset differently
if not DATASET_DIR.exists():
    DATASET_DIR = Path("/kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection")

TRAIN_CSV = DATASET_DIR / "train.csv"
TRAIN_IMG_DIR = DATASET_DIR / "train"

# Output metadata only
OUTPUT_DIR = Path("/kaggle/working/vinbigdata_subset_metadata")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)

print("DATASET_DIR:", DATASET_DIR)
print("TRAIN_CSV:", TRAIN_CSV)
print("TRAIN_IMG_DIR:", TRAIN_IMG_DIR)
print("TRAIN_CSV exists:", TRAIN_CSV.exists())
print("TRAIN_IMG_DIR exists:", TRAIN_IMG_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)

DATASET_DIR: /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection
TRAIN_CSV: /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train.csv
TRAIN_IMG_DIR: /kaggle/input/competitions/vinbigdata-chest-xray-abnormalities-detection/train
TRAIN_CSV exists: True
TRAIN_IMG_DIR exists: True
OUTPUT_DIR: /kaggle/working/vinbigdata_subset_metadata


In [2]:
# ============================================================
# 2. LOAD TRAIN ANNOTATION
# ============================================================

df = pd.read_csv(TRAIN_CSV)

print("Original train.csv shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

required_cols = [
    "image_id",
    "class_name",
    "class_id",
    "x_min",
    "y_min",
    "x_max",
    "y_max"
]

missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

print("\nClass distribution by annotation rows:")
display(df["class_name"].value_counts().reset_index().rename(
    columns={"index": "class_name", "class_name": "count"}
))

Original train.csv shape: (67914, 8)
Columns: ['image_id', 'class_name', 'class_id', 'rad_id', 'x_min', 'y_min', 'x_max', 'y_max']


,image_id,class_name,class_id,rad_id,x_min,y_min,x_max,y_max
0,50a418190bc3fb1ef1633bf9678929b3,No finding,14,R11,NaN,NaN,NaN,NaN
1,21a10246a5ec7af151081d0cd6d65dc9,No finding,14,R7,NaN,NaN,NaN,NaN
2,9a5094b2563a1ef3ff50dc5c7ff71345,Cardiomegaly,3,R10,691.0,1375.0,1653.0,1831.0
3,051132a778e61a86eb147c7c6f564dfe,Aortic enlargement,0,R10,1264.0,743.0,1611.0,1019.0
4,063319de25ce7edb9b1c6b8881290140,No finding,14,R10,NaN,NaN,NaN,NaN



Class distribution by annotation rows:


,count,count
0,No finding,31818
1,Aortic enlargement,7162
2,Cardiomegaly,5427
3,Pleural thickening,4842
4,Pulmonary fibrosis,4655
5,Nodule/Mass,2580
6,Lung Opacity,2483
7,Pleural effusion,2476
8,Other lesion,2203
9,Infiltration,1247


In [3]:
# ============================================================
# 3. IDENTIFY ABNORMAL AND NORMAL IMAGE IDS
# ============================================================

# Abnormal image:
# image has at least one annotation with class_id != 14
abnormal_image_ids = (
    df[df["class_id"] != NO_FINDING_CLASS_ID]["image_id"]
    .drop_duplicates()
    .tolist()
)

# Normal / No Finding image:
# all annotations of that image are class_id == 14
image_class_group = df.groupby("image_id")["class_id"].apply(list)

normal_image_ids = [
    image_id
    for image_id, class_ids in image_class_group.items()
    if all(cid == NO_FINDING_CLASS_ID for cid in class_ids)
]

print("Total unique images in train.csv:", df["image_id"].nunique())
print("Total abnormal images:", len(abnormal_image_ids))
print("Total normal / No Finding images:", len(normal_image_ids))

if len(normal_image_ids) < N_NORMAL:
    raise ValueError(
        f"Not enough normal images. Need {N_NORMAL}, "
        f"but found only {len(normal_image_ids)}."
    )

if len(abnormal_image_ids) != 4394:
    print(
        "Warning: abnormal image count is not 4,394. "
        f"Found: {len(abnormal_image_ids)}. "
        "This may still be okay depending on the dataset version or filtering."
    )

Total unique images in train.csv: 15000
Total abnormal images: 4394
Total normal / No Finding images: 10606


In [4]:
# ============================================================
# 4. SAMPLE 500 NORMAL + KEEP ALL ABNORMAL
# ============================================================

sampled_normal_image_ids = random.sample(normal_image_ids, N_NORMAL)

selected_image_ids = sorted(
    set(abnormal_image_ids) | set(sampled_normal_image_ids)
)

print("Selected abnormal images:", len(abnormal_image_ids))
print("Selected normal images:", len(sampled_normal_image_ids))
print("Total selected images:", len(selected_image_ids))

expected_total = len(abnormal_image_ids) + N_NORMAL

if len(selected_image_ids) != expected_total:
    raise ValueError(
        f"Selected image count mismatch. "
        f"Expected {expected_total}, got {len(selected_image_ids)}."
    )

selected_df = pd.DataFrame({
    "image_id": selected_image_ids
})

selected_df["subset_type"] = selected_df["image_id"].apply(
    lambda x: "abnormal" if x in set(abnormal_image_ids) else "normal"
)

display(selected_df["subset_type"].value_counts().reset_index().rename(
    columns={"index": "subset_type", "subset_type": "num_images"}
))

Selected abnormal images: 4394
Selected normal images: 500
Total selected images: 4894


,num_images,count
0,abnormal,4394
1,normal,500


In [5]:
# ============================================================
# 5. CREATE SUBSET ANNOTATION CSV
# ============================================================

subset_ann = df[df["image_id"].isin(selected_image_ids)].copy()

print("Subset annotation shape:", subset_ann.shape)
print("Number of images in subset annotation:", subset_ann["image_id"].nunique())

print("\nSubset class distribution by annotation rows:")
display(subset_ann["class_name"].value_counts().reset_index().rename(
    columns={"index": "class_name", "class_name": "count"}
))

# Kiểm tra ảnh selected nào không có annotation
selected_ids_set = set(selected_image_ids)
ann_ids_set = set(subset_ann["image_id"].unique())

selected_without_annotation = sorted(selected_ids_set - ann_ids_set)

print("Selected images without annotation:", len(selected_without_annotation))

if selected_without_annotation:
    pd.DataFrame({"image_id": selected_without_annotation}).to_csv(
        OUTPUT_DIR / "selected_without_annotation.csv",
        index=False
    )

Subset annotation shape: (37596, 8)
Number of images in subset annotation: 4894

Subset class distribution by annotation rows:


,count,count
0,Aortic enlargement,7162
1,Cardiomegaly,5427
2,Pleural thickening,4842
3,Pulmonary fibrosis,4655
4,Nodule/Mass,2580
5,Lung Opacity,2483
6,Pleural effusion,2476
7,Other lesion,2203
8,No finding,1500
9,Infiltration,1247


Selected images without annotation: 0


In [6]:
# ============================================================
# 6. CHECK IMAGE EXISTENCE WITHOUT COPYING
# ============================================================

existing_image_ids = []
missing_image_ids = []

for image_id in selected_image_ids:
    image_path = TRAIN_IMG_DIR / f"{image_id}.dicom"
    
    if image_path.exists():
        existing_image_ids.append(image_id)
    else:
        missing_image_ids.append(image_id)

print("Selected images:", len(selected_image_ids))
print("Existing DICOM images:", len(existing_image_ids))
print("Missing DICOM images:", len(missing_image_ids))

missing_images_df = pd.DataFrame({
    "image_id": missing_image_ids
})

missing_images_df.to_csv(
    OUTPUT_DIR / "missing_images.csv",
    index=False
)

# Nếu có ảnh thiếu, chỉ giữ ảnh thật sự tồn tại
selected_image_ids_final = sorted(existing_image_ids)

selected_df_final = selected_df[
    selected_df["image_id"].isin(selected_image_ids_final)
].copy()

subset_ann_final = subset_ann[
    subset_ann["image_id"].isin(selected_image_ids_final)
].copy()

print("Final selected images after image existence check:", len(selected_image_ids_final))
print("Final annotation rows:", len(subset_ann_final))

Selected images: 4894
Existing DICOM images: 4894
Missing DICOM images: 0
Final selected images after image existence check: 4894
Final annotation rows: 37596


In [7]:
# ============================================================
# 7. SAVE METADATA FILES
# ============================================================

selected_df_final.to_csv(
    OUTPUT_DIR / "selected_image_ids.csv",
    index=False
)

subset_ann_final.to_csv(
    OUTPUT_DIR / "subset_train_annotations.csv",
    index=False
)

# Lưu thêm abnormal và normal list riêng cho dễ kiểm tra
selected_df_final[selected_df_final["subset_type"] == "abnormal"].to_csv(
    OUTPUT_DIR / "abnormal_image_ids.csv",
    index=False
)

selected_df_final[selected_df_final["subset_type"] == "normal"].to_csv(
    OUTPUT_DIR / "normal_image_ids_500.csv",
    index=False
)

print("Saved metadata files to:", OUTPUT_DIR)

Saved metadata files to: /kaggle/working/vinbigdata_subset_metadata


In [8]:
# ============================================================
# 8. CREATE DATASET SUMMARY
# ============================================================

bbox_ann = subset_ann_final[
    subset_ann_final["class_id"] != NO_FINDING_CLASS_ID
].copy()

summary = {
    "random_seed": SEED,
    "target_normal_images": N_NORMAL,
    "original_total_annotation_rows": len(df),
    "original_total_unique_images": df["image_id"].nunique(),
    "original_abnormal_images": len(abnormal_image_ids),
    "original_normal_images": len(normal_image_ids),
    "selected_images_before_missing_check": len(selected_image_ids),
    "selected_images_after_missing_check": len(selected_image_ids_final),
    "selected_abnormal_images": selected_df_final[
        selected_df_final["subset_type"] == "abnormal"
    ]["image_id"].nunique(),
    "selected_normal_images": selected_df_final[
        selected_df_final["subset_type"] == "normal"
    ]["image_id"].nunique(),
    "subset_annotation_rows": len(subset_ann_final),
    "subset_bbox_rows_without_no_finding": len(bbox_ann),
    "subset_positive_images": bbox_ann["image_id"].nunique(),
    "subset_no_finding_images": selected_df_final[
        selected_df_final["subset_type"] == "normal"
    ]["image_id"].nunique(),
    "missing_dicom_images": len(missing_image_ids),
}

summary_df = pd.DataFrame([summary])

summary_df.to_csv(
    OUTPUT_DIR / "subset_summary.csv",
    index=False
)

display(summary_df)

,random_seed,target_normal_images,original_total_annotation_rows,original_total_unique_images,original_abnormal_images,original_normal_images,selected_images_before_missing_check,selected_images_after_missing_check,selected_abnormal_images,selected_normal_images,subset_annotation_rows,subset_bbox_rows_without_no_finding,subset_positive_images,subset_no_finding_images,missing_dicom_images
0,42,500,67914,15000,4394,10606,4894,4894,4394,500,37596,36096,4394,500,0


In [9]:
# ============================================================
# 9. CLASS DISTRIBUTION IN SUBSET
# ============================================================

class_distribution_rows = (
    subset_ann_final
    .groupby(["class_id", "class_name"])
    .agg(
        annotation_rows=("image_id", "count"),
        num_images=("image_id", "nunique")
    )
    .reset_index()
    .sort_values("annotation_rows", ascending=False)
)

class_distribution_rows.to_csv(
    OUTPUT_DIR / "subset_class_distribution.csv",
    index=False
)

display(class_distribution_rows)

,class_id,class_name,annotation_rows,num_images
0,0,Aortic enlargement,7162,3067
3,3,Cardiomegaly,5427,2300
11,11,Pleural thickening,4842,1981
13,13,Pulmonary fibrosis,4655,1617
8,8,Nodule/Mass,2580,826
7,7,Lung Opacity,2483,1322
10,10,Pleural effusion,2476,1032
9,9,Other lesion,2203,1134
14,14,No finding,1500,500
6,6,Infiltration,1247,613


In [10]:
# ============================================================
# 10. POSITIVE / NORMAL SUMMARY
# ============================================================

positive_image_ids = set(
    subset_ann_final[subset_ann_final["class_id"] != NO_FINDING_CLASS_ID]["image_id"].unique()
)

normal_image_ids_selected = set(
    selected_df_final[selected_df_final["subset_type"] == "normal"]["image_id"].unique()
)

positive_normal_summary = pd.DataFrame([{
    "total_subset_images": selected_df_final["image_id"].nunique(),
    "abnormal_images": len(positive_image_ids),
    "normal_no_finding_images": len(normal_image_ids_selected),
    "abnormal_ratio": len(positive_image_ids) / selected_df_final["image_id"].nunique(),
    "normal_ratio": len(normal_image_ids_selected) / selected_df_final["image_id"].nunique(),
}])

positive_normal_summary.to_csv(
    OUTPUT_DIR / "positive_normal_summary.csv",
    index=False
)

display(positive_normal_summary)

,total_subset_images,abnormal_images,normal_no_finding_images,abnormal_ratio,normal_ratio
0,4894,4394,500,0.897834,0.102166


In [11]:
# ============================================================
# 11. BASIC ANNOTATION VALIDATION FROM CSV ONLY
# No DICOM reading here
# ============================================================

ann = subset_ann_final.copy()
selected_ids = set(selected_df_final["image_id"].unique())
ann_ids = set(ann["image_id"].unique())

ann_not_in_selected = sorted(ann_ids - selected_ids)
selected_not_in_ann = sorted(selected_ids - ann_ids)

bbox_ann = ann[ann["class_id"] != NO_FINDING_CLASS_ID].copy()

invalid_bbox_basic = bbox_ann[
    (bbox_ann["x_min"].isna()) |
    (bbox_ann["y_min"].isna()) |
    (bbox_ann["x_max"].isna()) |
    (bbox_ann["y_max"].isna()) |
    (bbox_ann["x_max"] <= bbox_ann["x_min"]) |
    (bbox_ann["y_max"] <= bbox_ann["y_min"]) |
    (bbox_ann["x_min"] < 0) |
    (bbox_ann["y_min"] < 0)
].copy()

duplicate_annotation_rows = ann[ann.duplicated()].copy()

pd.DataFrame({"image_id": ann_not_in_selected}).to_csv(
    OUTPUT_DIR / "annotation_image_ids_not_in_selected.csv",
    index=False
)

pd.DataFrame({"image_id": selected_not_in_ann}).to_csv(
    OUTPUT_DIR / "selected_image_ids_without_annotation.csv",
    index=False
)

invalid_bbox_basic.to_csv(
    OUTPUT_DIR / "invalid_bbox_basic.csv",
    index=False
)

duplicate_annotation_rows.to_csv(
    OUTPUT_DIR / "duplicate_annotation_rows.csv",
    index=False
)

validation_summary_basic = pd.DataFrame([{
    "annotation_image_ids_not_in_selected": len(ann_not_in_selected),
    "selected_image_ids_without_annotation": len(selected_not_in_ann),
    "invalid_bbox_basic": len(invalid_bbox_basic),
    "duplicate_annotation_rows": len(duplicate_annotation_rows),
}])

validation_summary_basic.to_csv(
    OUTPUT_DIR / "annotation_validation_summary_basic.csv",
    index=False
)

display(validation_summary_basic)

,annotation_image_ids_not_in_selected,selected_image_ids_without_annotation,invalid_bbox_basic,duplicate_annotation_rows
0,0,0,0,0


In [12]:
# ============================================================
# 12. CREATE A README FOR THE METADATA SUBSET
# ============================================================

readme_text = f"""
# VinBigData Chest X-ray Metadata-only Subset

This subset contains metadata only. No DICOM images are copied.

## Dataset source

Kaggle competition:
VinBigData Chest X-ray Abnormalities Detection

## Subset rule

- Keep all abnormal images: class_id != {NO_FINDING_CLASS_ID}
- Randomly sample {N_NORMAL} normal / No Finding images: class_id == {NO_FINDING_CLASS_ID}
- Random seed: {SEED}

## Main files

1. selected_image_ids.csv
   - image_id
   - subset_type: abnormal or normal

2. subset_train_annotations.csv
   - annotation rows from original train.csv for selected images only

3. subset_summary.csv
   - summary statistics of the subset

4. subset_class_distribution.csv
   - class distribution in the subset

5. positive_normal_summary.csv
   - abnormal vs normal image count

6. annotation_validation_summary_basic.csv
   - basic annotation validation from CSV only

7. missing_images.csv
   - selected image ids whose DICOM file was not found in the Kaggle input folder

## Important note

DICOM images are not stored in this subset. To access images during training or validation,
use image_id to read files from:

{str(TRAIN_IMG_DIR)}/<image_id>.dicom

This design avoids Kaggle working directory storage errors and keeps the subset reproducible.
"""

with open(OUTPUT_DIR / "README_metadata_subset.md", "w", encoding="utf-8") as f:
    f.write(readme_text)

print(readme_text)


# VinBigData Chest X-ray Metadata-only Subset

This subset contains metadata only. No DICOM images are copied.

## Dataset source

Kaggle competition:
VinBigData Chest X-ray Abnormalities Detection

## Subset rule

- Keep all abnormal images: class_id != 14
- Randomly sample 500 normal / No Finding images: class_id == 14
- Random seed: 42

## Main files

1. selected_image_ids.csv
   - image_id
   - subset_type: abnormal or normal

2. subset_train_annotations.csv
   - annotation rows from original train.csv for selected images only

3. subset_summary.csv
   - summary statistics of the subset

4. subset_class_distribution.csv
   - class distribution in the subset

5. positive_normal_summary.csv
   - abnormal vs normal image count

6. annotation_validation_summary_basic.csv
   - basic annotation validation from CSV only

7. missing_images.csv
   - selected image ids whose DICOM file was not found in the Kaggle input folder

## Important note

DICOM images are not stored in this subset. T

In [13]:
# ============================================================
# 13. FINAL OUTPUT CHECK
# ============================================================

print("Files saved in:", OUTPUT_DIR)
print()

for path in sorted(OUTPUT_DIR.glob("*")):
    size_kb = path.stat().st_size / 1024
    print(f"{path.name:45s} {size_kb:10.2f} KB")

Files saved in: /kaggle/working/vinbigdata_subset_metadata

README_metadata_subset.md                           1.24 KB
abnormal_image_ids.csv                            180.24 KB
annotation_image_ids_not_in_selected.csv            0.01 KB
annotation_validation_summary_basic.csv             0.12 KB
duplicate_annotation_rows.csv                       0.06 KB
invalid_bbox_basic.csv                              0.06 KB
missing_images.csv                                  0.01 KB
normal_image_ids_500.csv                           19.55 KB
positive_normal_summary.csv                         0.14 KB
selected_image_ids.csv                            199.77 KB
selected_image_ids_without_annotation.csv           0.01 KB
subset_class_distribution.csv                       0.42 KB
subset_summary.csv                                  0.45 KB
subset_train_annotations.csv                     2931.89 KB


In [ ]:
# Cách dùng subset metadata sau này Khi train hoặc phân tích, anh chỉ cần đọc:

selected_df = pd.read_csv("/kaggle/working/vinbigdata_subset_metadata/selected_image_ids.csv")
ann = pd.read_csv("/kaggle/working/vinbigdata_subset_metadata/subset_train_annotations.csv")

# Khi cần đọc ảnh:
image_id = selected_df.iloc[0]["image_id"]
image_path = TRAIN_IMG_DIR / f"{image_id}.dicom"

# Tức là metadata xác định ảnh nào thuộc subset, còn ảnh gốc vẫn đọc trực tiếp từ Kaggle input. 
# Cách này phù hợp nhất cho workflow của anh vì nhẹ, tái lập được, và không bị lỗi hết dung lượng.